In [1]:
# --- PARAMÈTRES (toujours en tête du notebook) ---------------------------
CHEMIN_DONNEES = "portefeuille_vie.csv"
DATE_REF = "2025-01-01"     # début de la période d'observation (rachat_12m = rachat sur l'exercice 2025)
SEUIL_EFFECTIF = 100        # en dessous, un taux par segment est jugé peu fiable

# --- IMPORTS --------------------------------------------------------------
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)


# --- OUTIL D'AUTO-CONTRÔLE (ne pas modifier) ------------------------------
def verifier(libelle, test):
    """Affiche ✅ ou ❌ selon le résultat de `test` (une fonction sans argument)."""
    try:
        ok = bool(test())
    except Exception:
        ok = False
    print(("✅ " if ok else "❌ ") + libelle + ("" if ok else "   → à compléter ou à revoir"))

Python 3.14.6 | pandas 3.0.3 | numpy 2.4.6


---
# Atelier B — Charger et contrôler le portefeuille (≈ 28 min)
*Rappel théorique : les quatre gestes pandas, la matrice de travail X / y, le contrôle qualité (slides « pandas : les quatre gestes quotidiens » et « Contrôle qualité »).*
Objectif : **cartographier les défauts avant de calculer quoi que ce soit.** Le jeu de données contient des défauts volontaires.

### B1 ·  Charger et lire le schéma
Chargez le CSV en convertissant `date_effet` en vraie date, puis vérifiez dimensions et types : *10 secondes qui évitent des heures de débogage.*
<details><summary>💡 Indice</summary><code>pd.read_csv(chemin, parse_dates=["nom_colonne"])</code>. Les dimensions s'obtiennent avec <code>.shape</code> (lignes, colonnes).</details>

In [7]:
df_brut = pd.read_csv(CHEMIN_DONNEES, parse_dates=["date_effet"])

print("Dimensions (lignes, colonnes) :", df_brut.shape)
print(df_brut.dtypes)
df_brut.head()

Dimensions (lignes, colonnes) : (5030, 12)
id_contrat                      str
date_effet           datetime64[us]
generation                    int64
age                         float64
sexe                            str
produit                         str
canal                           str
encours                     float64
versement_initial           float64
rachat_hist                 float64
statut                          str
rachat_12m                    int64
dtype: object


,id_contrat,date_effet,generation,age,sexe,produit,canal,encours,versement_initial,rachat_hist,statut,rachat_12m
0,A-5517,2019-07-05,2019,43.0,F,UC,Courtier,26819.53,25504.67,0.0,en cours,0
1,A-4270,2020-02-13,2020,30.0,F,Euro,Banque,47585.07,40851.01,NaN,en cours,0
2,A-1788,2020-03-11,2020,74.0,F,Euro,Banque,27073.88,22768.65,0.0,racheté,1
3,A-3782,2019-08-08,2019,61.0,F,UC,Agent,22008.91,19949.96,0.0,en cours,0
4,A-4929,2024-07-08,2024,35.0,H,UC,Courtier,28615.40,26094.30,NaN,en cours,0


In [8]:
verifier("12 colonnes", lambda: df_brut.shape[1] == 12)
verifier("date_effet est bien une date", lambda: str(df_brut.date_effet.dtype).startswith("datetime"))

✅ 12 colonnes
✅ date_effet est bien une date


### B2 ·  Cartographier les trois familles de défauts
**Manquants, doublons, valeurs hors domaine.** Comptez-les, sans rien corriger encore.
- (a) Part de valeurs manquantes par colonne, de la plus touchée à la moins touchée
- (b) Nombre de contrats en double (clé `id_contrat`)
- (c) Âges hors de [18 ; 100] et encours négatifs ou nuls
<details><summary>💡 Indice</summary>(a) <code>df.isna().mean()</code> donne la part de manquants ; <code>.sort_values(ascending=False)</code> trie. (b) <code>df.duplicated("id_contrat").sum()</code>. (c) Un masque booléen : <code>(df.age &lt; 18) | (df.age &gt; 100)</code>.</details>

In [9]:
# (a) Part de manquants par colonne
manquants = df_brut.isna().mean().sort_values(ascending=False)
print((manquants * 100).round(1))

# (b) Doublons sur la clé id_contrat
nb_doublons = df_brut.duplicated("id_contrat").sum()

# (c) Valeurs hors domaine
nb_age_hors_domaine = ((df_brut.age < 18) | (df_brut.age > 100)).sum()
nb_encours_invalide = (df_brut.encours <= 0).sum()

print(f"\nDoublons : {nb_doublons} | âges hors domaine : {nb_age_hors_domaine} | encours <= 0 : {nb_encours_invalide}")

verifier("30 doublons détectés", lambda: nb_doublons == 30)
verifier("6 âges hors domaine", lambda: nb_age_hors_domaine == 6)
verifier("8 encours invalides", lambda: nb_encours_invalide == 8)
verifier("rachat_hist ≈ 41 % de manquants", lambda: 0.40 < manquants["rachat_hist"] < 0.42)

rachat_hist          41.0
canal                 2.0
age                   1.5
id_contrat            0.0
generation            0.0
date_effet            0.0
produit               0.0
sexe                  0.0
encours               0.0
versement_initial     0.0
statut                0.0
rachat_12m            0.0
dtype: float64

Doublons : 30 | âges hors domaine : 6 | encours <= 0 : 8
✅ 30 doublons détectés
✅ 6 âges hors domaine
✅ 8 encours invalides
✅ rachat_hist ≈ 41 % de manquants


### B3 ·  Le manquant n'est pas neutre
41 % des contrats n'ont pas de `rachat_hist`. **Défaillance du système ou donnée structurellement absente ?** Les deux situations ont des conséquences opposées sur le modèle. Calculez la part de manquants **par génération** pour trancher.
<details><summary>💡 Indice</summary><code>df.rachat_hist.isna()</code> donne un booléen par contrat ; <code>.groupby(df.generation).mean()</code> en fait une part par génération.</details>

In [11]:
part_manquants_gen = df.rachat_hist.isna().groupby(df.generation).mean()
print((part_manquants_gen * 100).round(1))

verifier("100 % de manquants pour la génération 2024", lambda: part_manquants_gen[2024] == 1.0)
verifier("moins de 15 % de manquants pour 2015-2022", lambda: part_manquants_gen.loc[2015:2022].max() < 0.15)

generation
2015      6.1
2016      9.5
2017      5.6
2018      9.1
2019      7.7
2020      5.9
2021      8.0
2022      9.7
2023    100.0
2024    100.0
Name: rachat_hist, dtype: float64
✅ 100 % de manquants pour la génération 2024
✅ moins de 15 % de manquants pour 2015-2022



**Lecture métier — proposition de corrigé** : les générations 2023 et 2024 sont manquantes à 100 % : c'est une **absence structurelle** (contrat trop récent pour avoir 24 mois d'historique), pas un défaut de qualité. Pour 2015-2022, 6 à 10 % de manquants subsistent : **défaut d'alimentation probable**, à remonter à l'IT.
Traitement : pour les contrats récents, créer un indicateur « historique non disponible » plutôt que d'imputer 0 (imputer 0 reviendrait à affirmer « aucun rachat passé ») ; pour les autres, enquêter sur la source avant de choisir.


### B4 ·  Le piège du filtre « contrats en cours » (et la fuite de cible)
Un collègue propose de calculer le taux de rachat sur *« les contrats en cours »* (`statut == "en cours"`). Testez. Puis croisez `statut` et `rachat_12m` pour comprendre.
<details><summary>💡 Indice</summary><code>pd.crosstab(df.colonne_lignes, df.colonne_colonnes)</code> compte les effectifs de chaque combinaison.</details>

In [12]:
actifs = df[df.statut == "en cours"]

taux_biaise = actifs.rachat_12m.mean()          # taux de rachat calculé sur ces contrats
tableau = pd.crosstab(df.statut, df.rachat_12m)     # statut en lignes, rachat_12m en colonnes

print("Taux de rachat sur les contrats 'en cours' :", taux_biaise)
print(tableau)

verifier("le taux « biaisé » vaut exactement 0", lambda: taux_biaise == 0)

Taux de rachat sur les contrats 'en cours' : 0.0
rachat_12m     0    1
statut               
en cours    4785    0
racheté        0  215
✅ le taux « biaisé » vaut exactement 0



**Lecture — proposition de corrigé** : `statut` est observé **au 31/12/2025**, c'est-à-dire *après* l'événement à prédire. Tous les contrats rachetés ont le statut « racheté » : filtrer sur « en cours » **supprime mécaniquement tous les rachats** et donne 0 %. Le tableau croisé est une diagonale parfaite : `statut` *est* la cible sous un autre nom.
Conséquence : `statut` ne doit **jamais** servir de variable explicative ni de filtre de population ; la population correcte est « contrats en cours **au 01/01/2025** ». C'est l'équivalent d'un calcul de taux dont l'exposition ne contient que des survivants.
